# Latent Component Gaussian Process (LCGP) — Replicated 1D Illustration

This notebook demonstrates the performance and diagnostic behavior of the **Latent Component Gaussian Process (LCGP)** model on 1D functional data. The goal is to show how LCGP uses repeated observations at the same input locations to learn a multi-output function efficiently.

Notice

1. **What replication means** in this setting: multiple noisy observations at the same input value $x$.
2. **Replication helps**: repeated measurements reduce uncertainty about the local mean and help estimate noise.
3. **LCGP models**: a multi-output function using a small number of latent Gaussian processes.


## Experiment cases

- **Case 1**: Uniform-ish replication across the input space.
- **Case 2**: Skewed replication, with many more repeats in one interval.
- **Case 3**: Hotspot replication, with very high counts at a few specific locations.

Set `CASE` and `PLOT_MODE` in the configuration cell to switch between these views.


## Imports

Use the following to install `lcgp`, `matplotlib`, and `tensorflow-probability` extras (required for this notebook.)

In [ ]:
# !pip install -U lcgp matplotlib tensorflow-probability[tf]

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from pathlib import Path

from call_model import LCGPRun
from lcgp import evaluation

rng = np.random.default_rng(42)

## True Function

The input is one-dimensional, $x \in [0,1]$, but the output has three dimensions:

$$
y(x) = \begin{bmatrix} f_1(x) \\ f_2(x) \\ f_3(x) \end{bmatrix}.
$$

This is a simple function for a multi-output simulator or experiment. Each output dimension has a different shape, but all three are driven by the same input $x$.

Because this is synthetic data, we know the true function. Later, we compare the LCGP predictions against this truth to see whether the model recovered the correct pattern.

In [ ]:
def f_true(x):
    x = np.asarray(x, dtype=np.float64)
    f1 = 0.8 + 0.3 * np.sin(2 * np.pi * x) + 0.2 * x
    f2 = 0.3 + 0.5 * np.cos(2 * np.pi * x)
    f3 = -0.4 - (x - 0.5) ** 2 + 0.2 * np.sin(4 * np.pi * x)
    return np.vstack([f1, f2, f3])

## Data Generation Functions

Each function below creates a replicated training dataset. The important distinction is between:

- **Unique input locations**: the distinct $x$ values where we collect data.
- **Total observations**: the full dataset after repeated measurements are included.

For example, if $x=0.25$ is measured 10 times, that is **one unique location** but **ten observations**. The replicated LCGP method can exploit this structure instead of treating every repeated row as completely separate.

The three functions differ only in how they assign replication counts:

| Function | Main idea | 
|---|---|
| `make_rep_data` | roughly even replication
| `make_rep_data_skewed` | many repeats in one region
| `make_rep_data_hotspots` | many repeats at a few points

The output convention is:

- `xtrain`: shape `(N, 1)`, where `N` is the total number of observations.
- `ytrain`: shape `(p, N)`, where `p=3` is the number of output dimensions.
- `xtest`: dense grid used for plotting predictions.
- `ytrue`: true outputs on the test grid.


In [ ]:
def make_rep_data(n_unique=12, rep_choices=(1, 2, 3, 4), noise_std=(0.05, 0.08, 0.10), rng=rng):
    """Case 1: Uniform-ish replication."""
    x_unique = np.linspace(0.0, 1.0, n_unique, dtype=np.float64)
    r = rng.choice(rep_choices, size=n_unique, replace=True)

    xs, ys = [], []
    for i, xi in enumerate(x_unique):
        yi_true = f_true([xi])[:, 0]
        for _ in range(int(r[i])):
            eps = rng.normal(0, noise_std, size=3).astype(np.float64)
            xs.append([xi])
            ys.append(yi_true + eps)

    xtrain = np.array(xs, dtype=np.float64)
    ytrain = np.array(ys, dtype=np.float64).T
    xtest  = np.linspace(0.0, 1.0, 400, dtype=np.float64)[:, None]
    ytrue  = f_true(xtest[:, 0])
    return xtrain, ytrain, xtest, ytrue


def make_rep_data_skewed(n_unique=40, heavy_region=(0.20, 0.45),
                         light_rep_choices=(1, 2), heavy_rep_choices=(8, 12, 16, 20),
                         noise_std=(0.05, 0.08, 0.10), rng=rng):
    """Case 2: Skewed replication — heavy in a contiguous subregion."""
    x_unique = np.linspace(0.0, 1.0, n_unique, dtype=np.float64)

    xs, ys = [], []
    for xi in x_unique:
        choices = heavy_rep_choices if (heavy_region[0] <= xi <= heavy_region[1]) else light_rep_choices
        r = int(rng.choice(choices))
        yi_base = f_true([xi])[:, 0]
        for _ in range(r):
            eps = np.array([rng.normal(0, s) for s in noise_std], dtype=np.float64)
            xs.append([xi])
            ys.append(yi_base + eps)

    xtrain = np.array(xs, dtype=np.float64)
    ytrain = np.array(ys, dtype=np.float64).T
    xtest  = np.linspace(0.0, 1.0, 400, dtype=np.float64)[:, None]
    ytrue  = f_true(xtest[:, 0])
    return xtrain, ytrain, xtest, ytrue


def make_rep_data_hotspots(n_unique=50,
                           hotspots=((0.15, 10, 15), (0.50, 18, 25), (0.80, 12, 20)),
                           base_rep_choices=(1,), noise_std=(0.05, 0.08, 0.10), rng=rng):
    """Case 3: Hotspot replication — very high counts at specific locations."""
    x_unique = np.linspace(0.0, 1.0, n_unique, dtype=np.float64)
    hotspot_idx = {np.argmin(np.abs(x_unique - x0)): (lo, hi) for (x0, lo, hi) in hotspots}

    xs, ys = [], []
    for i, xi in enumerate(x_unique):
        if i in hotspot_idx:
            lo, hi = hotspot_idx[i]
            r = int(rng.integers(lo, hi + 1))
        else:
            r = int(rng.choice(base_rep_choices))
        yi_base = f_true([xi])[:, 0]
        for _ in range(r):
            eps = np.array([rng.normal(0, s) for s in noise_std], dtype=np.float64)
            xs.append([xi])
            ys.append(yi_base + eps)

    xtrain = np.array(xs, dtype=np.float64)
    ytrain = np.array(ys, dtype=np.float64).T
    xtest  = np.linspace(0.0, 1.0, 400, dtype=np.float64)[:, None]
    ytrue  = f_true(xtest[:, 0])
    return xtrain, ytrain, xtest, ytrue

## Configuration

Use this cell to control the full experiment sweep.

The notebook now runs every requested option automatically:

- `CASES = (1, 2, 3)`
- `SUBMETHODS = ('rep', 'full')`
- `PLOT_MODES = ('y', 'g')`

For each `(CASE, SUBMETHOD)` pair, the notebook:

1. Generates the corresponding replicated dataset.
2. Trains one LCGP model.
3. Computes predictions and diagnostics.
4. Saves both output plots:
   - `lcgp_output.png`
   - `lcgp_latent.png`

Results are written under directories such as:

- `./results_figure_rep_uniform/`
- `./results_figure_full_skewed/`
- `./results_figure_rep_hotspots/`


In [ ]:
CASES = (1, 2, 3)              # run all replication-pattern cases
SUBMETHODS = ('rep', 'full')   # run both fitting methods
PLOT_MODES = ('y', 'g')        # save both output and latent-GP plots

BASE_RNG_SEED = 42             # reproducible but independent runs per option
RUNNO_PREFIX = 'rep_1d'

## Run All Options

The following cells define reusable helpers and then run the full sweep across every configured option.

The loop trains once for each `(CASE, SUBMETHOD)` pair. It then generates each requested `PLOT_MODE` from the same fitted model, avoiding unnecessary duplicate training.


In [ ]:

def make_case_data(case, rng):
    """Generate data for one replication case."""
    if case == 1:
        suffix = 'uniform'
        xtrain, ytrain, xtest, ytrue = make_rep_data(
            n_unique=16, rep_choices=(1, 2, 3, 4, 5),
            noise_std=(0.05, 0.08, 0.10), rng=rng
        )
    elif case == 2:
        suffix = 'skewed'
        xtrain, ytrain, xtest, ytrue = make_rep_data_skewed(
            n_unique=40, heavy_region=(0.20, 0.45),
            light_rep_choices=(1, 2), heavy_rep_choices=(8, 12, 16, 20),
            noise_std=(0.05, 0.08, 0.10), rng=rng
        )
    elif case == 3:
        suffix = 'hotspots'
        xtrain, ytrain, xtest, ytrue = make_rep_data_hotspots(
            n_unique=50, hotspots=((0.15, 10, 15), (0.50, 18, 25), (0.80, 12, 20)),
            base_rep_choices=(1,), noise_std=(0.05, 0.08, 0.10), rng=rng
        )
    else:
        raise ValueError('case must be 1, 2, or 3')

    return suffix, xtrain, ytrain, xtest, ytrue


def print_replication_summary(case, xtrain, ytrain):
    """Print summary statistics for a generated replicated dataset."""
    print(f'Case {case} | N={xtrain.shape[0]} total obs | p={ytrain.shape[0]} outputs')

    x_unique_vals, rep_counts = np.unique(xtrain[:, 0], return_counts=True)
    print(f'Unique x locations: {len(x_unique_vals)}')
    print(f'Replication count summary: min={rep_counts.min()}, mean={rep_counts.mean():.2f}, max={rep_counts.max()}')
    print('First few (x, replication count) pairs:')
    for xi, ri in list(zip(x_unique_vals, rep_counts))[:8]:
        print(f'  x={xi:.3f}, r={ri}')


def fit_lcgp(case, submethod, xtrain, ytrain, xtest, ytrue):
    """Build and train one LCGP model."""
    data = {
        'xtrain': xtrain,
        'xtest':  xtest,
        'ytrain': ytrain,
        'ytest':  ytrue,
        'ytrue':  ytrue
    }

    modelrun = LCGPRun(
        runno=f'{RUNNO_PREFIX}_case{case}_{submethod}',
        data=data,
        num_latent=3,
        var_threshold=None,
        submethod=submethod,
        diag_error_structure=[1, 1, 1],
        robust_mean=True,
    )
    modelrun.define_model()

    t0 = time.time()
    modelrun.train()
    t1 = time.time()

    print(f'Training time: {t1 - t0:.2f}s')
    return modelrun


def print_diagnostics(modelrun):
    """Print basis, fitted-parameter, and replication diagnostics."""
    mdl = modelrun.model

    print('=== BASIS ===')
    print(f'diag_D values: {mdl.diag_D.numpy()}')
    print(f'phi^T @ phi diagonal: {np.diag(mdl.phi.numpy().T @ mdl.phi.numpy())}')

    print('\n=== FITTED PARAMETERS ===')
    lLmb, lLmb0, lsigma2s, lnugGPs = mdl.get_param()
    for k in range(lLmb.shape[0]):
        print(f'  Lengthscale component {k}: {lLmb[k].numpy()}')
    print(f'Noise log-var (lsigma2s): {lsigma2s.numpy()}')
    print(f'Noise std (fitted): {np.sqrt(np.exp(lsigma2s.numpy()))}')
    print(f'Noise std (true):   [0.05, 0.08, 0.10]')

    print('\n=== REPLICATION STATS ===')
    if mdl.submethod == 'rep':
        r = mdl.r.numpy()
        print(f'Unique locations n: {len(r)}')
        print(f'Total samples N:    {np.sum(r)}')
        print(f'Avg / Min / Max reps: {np.mean(r):.2f} / {np.min(r)} / {np.max(r)}')
    else:
        print("No replication stats for 'full' submethod.")

def evaluate_predictions(ytrue, predmean, yconfvar):
    """Compute and print evaluation metrics."""
    rmse   = evaluation.rmse(ytrue, predmean)
    nrmse  = evaluation.normalized_rmse(ytrue, predmean)
    pcover, pwidth = evaluation.intervalstats(ytrue, predmean, yconfvar)
    dss    = evaluation.dss(ytrue, predmean, yconfvar, use_diag=True)

    metrics = {
        'RMSE': rmse,
        'NRMSE': nrmse,
        '95% PI coverage': pcover,
        '95% PI width': pwidth,
        'DSS': dss,
    }

    print(f'RMSE:            {rmse:.4f}')
    print(f'NRMSE:           {nrmse:.4f}')
    print(f'95% PI coverage: {pcover:.3f}')
    print(f'95% PI width:    {pwidth:.4f}')
    print(f'DSS:             {dss:.4f}')
    return metrics


def plot_results(case, plot_mode, results_fig_path, modelrun, xtrain, ytrain, xtest, ytrue, predmean, yconfvar):
    """Save either output-space predictions or latent-GP plots."""
    mdl = modelrun.model
    order_test  = np.argsort(xtest[:, 0])
    order_train = np.argsort(xtrain[:, 0])

    if plot_mode == 'g':
        _ = mdl.predict(x0=xtest, return_fullcov=False)
        ghat_test = np.asarray(mdl.ghat)
        gstd_test = np.sqrt(np.asarray(mdl.gvar))
        ghat_tr   = np.asarray(mdl.mks)
        if mdl.submethod == 'rep':
            x_tr_unique = np.asarray(mdl.x_unique)
        elif mdl.submethod == 'full':
            x_tr_unique = np.asarray(mdl.x)
        order_unique = np.argsort(x_tr_unique[:, 0])
        q = ghat_test.shape[0]
        fig, axes = plt.subplots(q, 1, figsize=(10, 1.9 * q), sharex=True)
        if q == 1:
            axes = [axes]

        x_test_sorted = xtest[order_test, 0]
        x_tr_sorted   = x_tr_unique[order_unique, 0]

        for k, ax in enumerate(axes):
            m = ghat_test[k, order_test]
            s = gstd_test[k, order_test]
            ax.plot(x_test_sorted, m, lw=1.8, label=fr'$g_{{{k+1}}}(x)$ mean')
            ax.fill_between(x_test_sorted, m - 1.96 * s, m + 1.96 * s,
                            alpha=0.22, label='95% band')
            ax.scatter(x_tr_sorted, ghat_tr[k, order_unique],
                       s=12, alpha=0.65, label='train pts')
            ax.set_ylabel(fr'$g_{{{k+1}}}(x)$')
            ax.legend(loc='best', fontsize=9)

        axes[-1].set_xlabel('x')
        plt.suptitle(f'Latent GPs — Case {case}', y=1.01)
        plt.tight_layout()
        outfile = Path(results_fig_path) / 'lcgp_latent.png'

    elif plot_mode == 'y':
        fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
        for i in range(3):
            ax[i].scatter(xtrain[order_train, 0], ytrain[i, order_train],
                          s=12, alpha=0.65,
                          label='replicates' if i == 0 else None)
            ax[i].plot(xtest[order_test, 0], ytrue[i, order_test],
                       lw=1.8, label='true' if i == 0 else None)
            ax[i].plot(xtest[order_test, 0], predmean[i, order_test],
                       lw=1.5, label='LCGP mean' if i == 0 else None)
            lo = predmean[i, order_test] - 1.96 * np.sqrt(yconfvar[i, order_test])
            hi = predmean[i, order_test] + 1.96 * np.sqrt(yconfvar[i, order_test])
            ax[i].fill_between(xtest[order_test, 0], lo, hi,
                               alpha=0.22,
                               label='95% credible band' if i == 0 else None)
            ax[i].set_ylabel(f'$f_{i+1}(x)$')

        ax[-1].set_xlabel('x')
        ax[0].legend(loc='best', fontsize=9)
        plt.suptitle(f'LCGP Predictions — Case {case}', y=1.01)
        plt.tight_layout()
        outfile = Path(results_fig_path) / 'lcgp_output.png'

    else:
        raise ValueError("Choose plot_mode from 'g' or 'y'")

    plt.savefig(outfile, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {outfile}')
    return outfile


In [ ]:
all_results = []

for case in CASES:
    for submethod in SUBMETHODS:
        print('=' * 80)
        print(f'RUNNING CASE={case}, SUBMETHOD={submethod}')
        print('=' * 80)

        # Use a deterministic but different RNG stream for each case/submethod.
        submethod_offset = 0 if submethod == 'rep' else 1000
        rng = np.random.default_rng(BASE_RNG_SEED + 100 * case + submethod_offset)

        case_suffix, xtrain, ytrain, xtest, ytrue = make_case_data(case, rng)
        results_fig_path = Path(f'./results_figure_{submethod}_{case_suffix}')
        results_fig_path.mkdir(parents=True, exist_ok=True)

        print_replication_summary(case, xtrain, ytrain)

        modelrun = fit_lcgp(case, submethod, xtrain, ytrain, xtest, ytrue)
        predmean, ypredvar, yconfvar = modelrun.predict(return_fullcov=False)

        print_diagnostics(modelrun)
        metrics = evaluate_predictions(ytrue, predmean, yconfvar)

        plot_files = []
        for plot_mode in PLOT_MODES:
            plot_files.append(
                plot_results(
                    case=case,
                    plot_mode=plot_mode,
                    results_fig_path=results_fig_path,
                    modelrun=modelrun,
                    xtrain=xtrain,
                    ytrain=ytrain,
                    xtest=xtest,
                    ytrue=ytrue,
                    predmean=predmean,
                    yconfvar=yconfvar,
                )
            )

        all_results.append({
            'case': case,
            'submethod': submethod,
            'results_fig_path': str(results_fig_path),
            'metrics': metrics,
            'plot_files': [str(p) for p in plot_files],
        })

print('=' * 80)
print('COMPLETED ALL OPTIONS')
print('=' * 80)
for result in all_results:
    print(
        f"CASE={result['case']}, SUBMETHOD={result['submethod']} | "
        f"RMSE={result['metrics']['RMSE']:.4f}, "
        f"NRMSE={result['metrics']['NRMSE']:.4f} | "
        f"plots={result['plot_files']}"
    )
